In [1]:
!pip install xgboost imbalanced-learn joblib

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [20]:
import pandas as pd

df = pd.read_csv("Medical_appointment_data.csv")

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Medical_appointment_data.csv to Medical_appointment_data (1).csv


In [7]:
import os
print(os.listdir())

['.config', 'Medical_appointment_data.csv', 'Medical_appointment_data (1).csv', 'sample_data']


In [9]:
import pandas as pd

df = pd.read_csv("Medical_appointment_data.csv")

print(df.shape)
df.head()

(109593, 26)


,specialty,appointment_time,gender,no_show,disability,place,appointment_shift,age,under_12_years_old,over_60_years_old,...,storm_day_before,rain_intensity,heat_intensity,appointment_date_continuous,Hipertension,Diabetes,Alcoholism,Handcap,Scholarship,SMS_received
0,psychotherapy,17,F,yes,intellectual,Lake Marvinville,afternoon,9.0,1,0,...,1,no_rain,warm,2020-01-01,0,0,0,0,0,0
1,NaN,7,M,no,intellectual,ITAPEMA,morning,11.0,1,0,...,1,no_rain,cold,2020-01-01,0,0,0,0,0,0
2,speech therapy,16,M,no,intellectual,ITAJAÍ,afternoon,8.0,1,0,...,1,no_rain,warm,2020-01-01,0,0,0,0,0,0
3,speech therapy,14,M,yes,intellectual,Sarahside,afternoon,9.0,1,0,...,1,moderate,mild,2020-01-01,0,0,0,0,0,1
4,physiotherapy,8,M,no,motor,ITAJAÍ,morning,NaN,0,0,...,1,no_rain,mild,2020-01-01,0,0,0,0,0,0


In [10]:
df.columns.tolist()

['specialty',
 'appointment_time',
 'gender',
 'no_show',
 'disability',
 'place',
 'appointment_shift',
 'age',
 'under_12_years_old',
 'over_60_years_old',
 'patient_needs_companion',
 'average_temp_day',
 'average_rain_day',
 'max_temp_day',
 'max_rain_day',
 'rainy_day_before',
 'storm_day_before',
 'rain_intensity',
 'heat_intensity',
 'appointment_date_continuous',
 'Hipertension',
 'Diabetes',
 'Alcoholism',
 'Handcap',
 'Scholarship',
 'SMS_received']

In [11]:
df.isnull().sum()


,0
specialty,20127
appointment_time,0
gender,0
no_show,0
disability,16601
place,11539
appointment_shift,0
age,22960
under_12_years_old,0
over_60_years_old,0


In [12]:
for col in df.columns:

    if df[col].dtype == "object":
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

print("Missing values handled successfully")

Missing values handled successfully


/tmp/ipykernel_4008/2579452237.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipykernel_4008/2579452237.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [21]:
df["no_show"].value_counts(dropna=False)

,count
no_show,
no,74761
yes,34832


In [26]:
df["no_show"].unique()

array(['yes', 'no'], dtype=object)

In [28]:
df["no_show"] = df["no_show"].map({
    "yes": 1,
    "no": 0
})

df["no_show"].value_counts()


,count
no_show,
0,74761
1,34832


In [29]:
for col in df.columns:

    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])

    else:
        df[col] = df[col].fillna(df[col].median())

print("Missing values handled")

Missing values handled


In [30]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col].astype(str))

print("Encoding completed")

Encoding completed


In [31]:
X = df.drop("no_show", axis=1)

y = df["no_show"]

print(X.shape)
print(y.shape)

(109593, 25)
(109593,)


In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (87674, 25)
Test: (21919, 25)


In [33]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [34]:
from sklearn.metrics import accuracy_score, f1_score

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("F1 Score:", f1_score(y_test, pred))

Accuracy: 0.7153611022400657
F1 Score: 0.47229975471538527


In [35]:
import joblib

joblib.dump(model, "no_show_model.pkl")

print("Model Saved")

Model Saved


In [36]:
import joblib

joblib.dump(model, "no_show_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [37]:
from google.colab import files
files.download("no_show_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
daily = df.groupby("appointment_date_continuous").size().reset_index()

daily.columns = ["date", "appointments"]

daily.head()

,date,appointments
0,0,25
1,1,33
2,2,18
3,3,16
4,4,15


In [39]:
daily["date"] = pd.to_datetime(daily["date"])

daily["day"] = daily["date"].dt.day
daily["month"] = daily["date"].dt.month
daily["weekday"] = daily["date"].dt.weekday

daily.head()

,date,appointments,day,month,weekday
0,1970-01-01 00:00:00.000000000,25,1,1,3
1,1970-01-01 00:00:00.000000001,33,1,1,3
2,1970-01-01 00:00:00.000000002,18,1,1,3
3,1970-01-01 00:00:00.000000003,16,1,1,3
4,1970-01-01 00:00:00.000000004,15,1,1,3


In [41]:
X = daily[["day", "month", "weekday"]]

y = daily["appointments"]

In [43]:
split = int(len(daily) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [44]:
from sklearn.ensemble import RandomForestRegressor

forecast_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

forecast_model.fit(X_train, y_train)

print("Forecast Model Trained")

Forecast Model Trained


In [45]:
from sklearn.ensemble import RandomForestRegressor

forecast_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

forecast_model.fit(X_train, y_train)

print("Forecast Model Trained")

Forecast Model Trained


In [46]:
joblib.dump(
    forecast_model,
    "demand_forecast_model.pkl"
)

print("Demand Forecast Model Saved")

Demand Forecast Model Saved


In [47]:
from google.colab import files

files.download("demand_forecast_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>